---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\valer\Desktop\Analiza Datelor Complexe\17. AI Avansat\echochamber-project-team3
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
from pathlib import Path
import pandas as pd

def project_root():
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        # rădăcina e folderul care conține ATÂT `data/` CÂT ȘI `notebooks/`
        if (parent / "data").is_dir() and (parent / "notebooks").is_dir():
            return parent
    raise FileNotFoundError("Nu am găsit rădăcina proiectului")

ROOT = project_root()
path = ROOT / "data" / "cleaned" / "corpus_youtube_sample.jsonl"

print("ROOT:", ROOT)
print("Cale:", path)
print("Există:", path.exists(), "| Mărime:", path.stat().st_size if path.exists() else "N/A")

corpus = pd.read_json(path, lines=True)
print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

ROOT: C:\Users\valer\Desktop\Analiza Datelor Complexe\17. AI Avansat\echochamber-project-team3
Cale: C:\Users\valer\Desktop\Analiza Datelor Complexe\17. AI Avansat\echochamber-project-team3\data\cleaned\corpus_youtube_sample.jsonl
Există: True | Mărime: 178652
420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [3]:
# modifica dupa preferinte
AXA_1 = "people_vs_elite"
AXA_2 = "national_identity"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [4]:
AXA_1_DEFINITION = """
people_vs_elite măsoară dacă textul construiește o opoziție între
"poporul autentic" (oameni cinstiți, muncitori, trădați, ignorați)
și o "elită" prezentată ca grup separat și ostil (politicieni, bogați,
sistem, "ei de sus", "cei care ne conduc", clasa politică, oligarhi).

Cheia este OPOZIȚIA explicită sau implicită între cele două categorii,
nu doar criticarea unei persoane sau a unui partid anume.

0 = absent
    Textul nu construiește această opoziție. Poate critica pe cineva,
    dar nu invocă "poporul" vs "elita" ca grupuri distincte.
    Ex: "nu îmi place politica lui X", "părerea mea e că legea e proastă".

1 = prezent
    Apare opoziția popor-elită, dar e o componentă printre altele
    sau e exprimată moderat / indirect.
    Ex: "politicienii nu ne ascultă", "oamenii simpli plătesc factura",
    "ei iau decizii fără noi".

2 = dominant
    Opoziția popor-elită este nucleul comentariului. Textul e construit
    în jurul ideii că un "noi" (poporul, oamenii cinstiți) e trădat,
    furat sau ignorat de un "ei" (clasa politică, sistemul, elita).
    Limbaj puternic moralizator sau dihotomic.
    Ex: "35 de ani de trădare națională", "ei fură, noi muncim",
    "poporul s-a săturat de mafia de la putere".
"""

AXA_2_DEFINITION = """
national_identity măsoară dacă textul invocă identitatea națională,
suveranitatea, demnitatea sau caracterul țării/poporului român
ca valoare centrală sau ca element amenințat.

Markere tipice: România, român/românesc, neam, popor, țară, suveranitate,
demnitate națională, tradiție, istorie, "cine suntem noi", patrie.

0 = absent
    Textul nu invocă identitatea națională. Poate vorbi despre politică
    sau societate fără a apela la apartenența națională.
    Ex: "economia merge prost", "votez cu cine îmi place".

1 = prezent
    Apare o referință la identitate națională, dar e marginală
    sau folosită descriptiv, nu ca pilon al argumentului.
    Ex: "ca român, cred că...", "în țara asta nu merge nimic",
    "ne trebuie o Românie mai bună".

2 = dominant
    Identitatea națională este axul comentariului. Textul invocă
    explicit suveranitatea, demnitatea, "neamul românesc", "România
    adevărată" sau prezintă identitatea ca fiind amenințată / trădată /
    de salvat.
    Ex: "să construim o Românie suverană, curată și demnă",
    "neamul nostru merită mai mult", "trebuie să ne salvăm țara
    de trădători".
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [5]:
MINI_PROMPT = f"""
Ești un adnotator expert în analiza discursului politic românesc online.
Lucrezi pe comentarii reale de pe YouTube despre politica românească.
Aplici criterii fixe, nu interpretezi liber și nu adaugi nimic în afara schemei.

SARCINĂ:
Adnotează comentariul folosind două axe discursive și trei câmpuri descriptive.
Axele de adnotat sunt:
1. {AXA_1}
2. {AXA_2}

CÂMPURI:
target = ținta politică principală din comentariu (persoană, partid, instituție,
         categorie de actori, țară). Folosește un nume scurt și concret.
         Dacă există mai multe ținte, alege-o pe cea centrală.
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator /
       defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2

DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe. Dacă nu ești sigur, alege valoarea mai conservatoare
   (de ex. 1 în loc de 2, "neutru" în loc de "acuzator").
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
   Ex: "ce democrație frumoasă avem" cu ton evident sarcastic → stance="anti", tone="ironic".
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant. Dacă elementul apare
   marginal sau implicit → 1. Dacă este nucleul comentariului → 2.
6. Nu atribui direct o bulă discursivă. Nu folosi etichete ca "personalist-salvator",
   "anti-sistem", "conspiraționist" etc. Codifici doar câmpurile cerute.
7. Răspunde EXCLUSIV cu un obiect JSON valid, fără text înainte sau după,
   fără ```json, fără comentarii. Toate cheile trebuie prezente.

FORMAT JSON așteptat:
{{
  "target": "string",
  "stance": "pro" | "anti" | "neutru" | "ambiguu" | "none",
  "tone": "acuzator" | "ironic" | "mobilizator" | "defensiv" | "afectiv" | "neutru",
  "{AXA_1}": 0 | 1 | 2,
  "{AXA_2}": 0 | 1 | 2
}}
"""

print(MINI_PROMPT)



Ești un adnotator expert în analiza discursului politic românesc online.
Lucrezi pe comentarii reale de pe YouTube despre politica românească.
Aplici criterii fixe, nu interpretezi liber și nu adaugi nimic în afara schemei.

SARCINĂ:
Adnotează comentariul folosind două axe discursive și trei câmpuri descriptive.
Axele de adnotat sunt:
1. people_vs_elite
2. national_identity

CÂMPURI:
target = ținta politică principală din comentariu (persoană, partid, instituție,
         categorie de actori, țară). Folosește un nume scurt și concret.
         Dacă există mai multe ținte, alege-o pe cea centrală.
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator /
       defensiv / afectiv / neutru
people_vs_elite = 0 / 1 / 2
national_identity = 0 / 1 / 2

DEFINIȚII:

people_vs_elite măsoară dacă textul construiește o opoziție între
"poporul autentic" (oameni cinstiți, muncitori, trădați, ignorați)
și o "elită" pre

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [6]:
TESTS = corpus.sample(5, random_state=42)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
145,yt_BfvZ8QcVBKc_Ugyog0iMEqAX5zIQb4R4AaABAg,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Da, si eu cred ca serviciile ucrainene au fost..."
334,yt_qkGhsJFft00_UgzJHKCoGkwtTJ3uOBh4AaABAg,digi24hd56,În fața ta cu Emil Hurezeanu: „Ar fi un coșmar...,Ce mă supără pe mine oamenii ăștia care sunt a...
175,yt_KqrUotq1Obs_Ugy6tYmdfH0T2THArnB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pace și prosperitate ( 28.10...,Bunul Dumnezeu să îl protejeze pe președintele...
369,yt_vkP6FdP9iX0_Ugwj3HojTt6JikJVtjd4AaABAg,turcescu111,Orientul Mijlociu în flăcări,Nicușor merge pe lângă covor pentru că nu are ...
416,yt_Sj4fQKlMOro_UgyNWIwNDtUeTTCrLkl4AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Trebuie susținută aceasta femeie!!!!! Acesta a...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [7]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [8]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [9]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Da, si eu cred ca serviciile ucrainene au fost inplicate in alegerile noastre. Am convingerea ca serviciile noastre sunt infiltrate de serviciile ucrainene

OUTPUT MODEL:
```json
{
  "target": "serviciile ucrainene",
  "stance": "anti",
  "tone": "acuzator",
  "people_vs_elite": 0,
  "national_identity": 1
}
```
COMENTARIU:
Ce mă supără pe mine oamenii ăștia care sunt așa de siguri când spun ca Iranul nu mai are arsenal militar. De unde mama dracului știu ei treburile astea. Noi bănuim ca primesc arme din Rusia bolshevica și China comunistă. Poate și Brazilia dar iarăși e o bănuială.

OUTPUT MODEL:
```json
{
  "target": "Iran",
  "stance": "anti",
  "tone": "acuzator",
  "people_vs_elite": 0,
  "national_identity": 0
}
```
COMENTARIU:
Bunul Dumnezeu să îl protejeze pe președintele nostru Călin Georgescu ❤️❤️❤️

OUTPUT MODEL:
```json
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "people_vs_elite": 0,
  "national_identity": 0
}
```
COMENTARIU:
Nic

## Pasul 6 — Interpretare scurtă

**Axele alese:** `people_vs_elite` și `national_identity`.

**De ce:** Lucrez pe bula *personalist-salvator*, care se sprijină pe două
elemente discursive: opoziția dintre „poporul autentic" și „elita coruptă"
(de unde `people_vs_elite`) și invocarea identității naționale ca valoare
trădată sau de salvat (de unde `national_identity`). Aceste două axe se
co-activează des în bula mea, dar discriminează bine față de celelalte bule
(pro-european, anti-sistem, conspiraționist), care folosesc combinații diferite.

**JSON corect?** Da, modelul a returnat JSON valid pe toate cele 5 comentarii,
cu toate cele 5 câmpuri cerute (`target`, `stance`, `tone`, `people_vs_elite`,
`national_identity`). Singura particularitate: răspunsul vine împachetat în
ghilimele de cod (` ```json ... ``` `), deși promptul cere JSON pur — va trebui
curățat înainte de `json.loads()`.

**Cea mai mare problemă:** Discriminarea între `1` (prezent) și `2` (dominant)
pe `national_identity`. De exemplu, comentariul despre „serviciile ucrainene
infiltrate" a primit `national_identity=1`, deși tema suveranității e centrală;
iar comentariul despre Iran a primit `national_identity=1`, deși axul lui e
geopolitica externă, nu identitatea românească. Modelul pare să marcheze ca
„prezent" orice referință națională, fără să verifice dacă e nucleul argumentului.

**Ce aș schimba în prompt:**
1. Aș adăuga 2–3 **exemple few-shot** pentru fiecare nivel (0/1/2), ca să fixez
   pragul între „prezent" și „dominant".
2. Aș întări **Regula 7** cu un exemplu negativ explicit: „NU împacheta în
````````json ```` — răspunde direct cu `{`". Modelul ignoră regula altfel.
3. Pentru `national_identity`, aș adăuga un criteriu de excludere: comentariile
   despre politică externă (Iran, Ucraina) primesc `national_identity=1` doar
   dacă fac referire la *România* sau la *identitatea românească*, nu doar la
   alte state.
````
